# Задание 1
Реализуйте базовый класс Account, который моделирует поведение банковского счёта. Этот класс должен не только выполнять базовые операции, но и вести детальный учёт всех действий, а также предоставлять аналитику по истории операций.

Этап 1. Реализация базового класса Checkingсса Account
Класс должен быть инициализирован с параметрами:
* account_holder (str) — имя владельца счёта;
* balance (float, по умолчанию 0) — начальный баланс счёта, не может быть отрицательным.

Атрибуты:
* _account_counter — приватный атрибут для хранения количества созданных счетов. Отсчет начинается с 1000;
* holder — хранит имя владельца;Checking
* account_number — хранит номер счёта;
* _balance — приватный атрибут для хранения текущего баланса;
* operations_history — список или другая структура для хранения истории операций.

Важно: каждая операция должна храниться не просто как число, а как структурированная информация, например, словарь, кортеж или класс. Минимальный набор данных для операции: тип операции (‘deposit’ или ‘withdraw’), сумма, дата и время операции, текущий баланс после операции, статус (‘success’ или ‘fail’).

Этап 2. Реализация методов

init(self, account_holder, balance=0) — конструктор. Обратите внимание, что в конструкторе должен автоматически формироваться номер счёта в формате ‘ACC-XXXX’, где XXXX — порядковый номер счёта;
deposit(self, amount) — метод для пополнения счёта:
* принимает сумму (должна быть положительной), попытка положить отрицательную сумму, должна вызывать исключение;
* в случае успеха обновляет баланс и добавляет запись в историю операций.
withdraw(self, amount) — метод для снятия средств:
* принимает сумму (должна быть положительной);
* проверяет, достаточно ли средств на счёте, если нет — операция не проходит, но ее попытка с статусом ‘fail’ все равно фиксируется в истории;
* в случае успеха обновляет баланс и добавляет запись в историю.
get_balance(self) — метод, который возвращает текущий баланс.
get_history(self) — метод, который возвращает историю операций.
Важно: продумайте, в каком формате его вернуть. Для работы с датой и временем используйте модуль datetime. Получить текущее время можно с помощью datetime.now().

In [12]:
from datetime import datetime


class Account:
    
    _account_counter = 1000

    def __init__(self, account_holder: str, balance: float = 0):
        if balance < 0:
            raise ValueError("Стартовый баланс не может быть отрицательным")

        self.holder = account_holder
        self.account_number = f"ACC-{Account._account_counter}"
        Account._account_counter += 1

        self._balance = float(balance)
        self.operations_history = []

    # Логирование эвентов
    def _add_operation(self, op_type: str, amount: float, status: str):
        self.operations_history.append({
            "type": op_type,
            "amount": amount,
            "datetime": datetime.now(),
            "balance_after": self._balance,
            "status": status
        })

    def deposit(self, amount: float):
        if amount <= 0:
            self._add_operation("deposit", amount, "fail")
            raise ValueError("Сумма пополнения должна быть положительной.")

        self._balance += amount
        self._add_operation("deposit", amount, "success")

    def withdraw(self, amount: float):
        if amount <= 0:
            self._add_operation("withdraw", amount, "fail")
            raise ValueError("Сумма снятия должна быть положительной.")

        if amount > self._balance:
            self._add_operation("withdraw", amount, "fail")
            raise ValueError("Недостаточно средств на счёте.")

        self._balance -= amount
        self._add_operation("withdraw", amount, "success")

    def get_balance(self):
        return self._balance

    def get_history(self):
        return self.operations_history

    def get_largest_operations(self, n):
        remaining = list(self.operations_history)
        result = []

        for _ in range(n):
            if not remaining:
                break
            biggest = remaining[0]
            for op in remaining:
                if op["amount"] > biggest["amount"]:
                    biggest = op
            result.append(biggest)
            remaining.remove(biggest)
        return result

In [13]:
# ТЕСТЫ ДЛЯ ПРОВЕРКИ
acc = Account("Иван Петров", 500)

acc.deposit(200)
acc.withdraw(100)

print(acc.holder)            # должно быть "Иван Петров"
print(acc.account_number)    # должно быть "ACC-1000"
print(acc.get_balance())     # должно быть "600.0"

for op in acc.get_history():
    print(op)


Иван Петров
ACC-1000
600.0
{'type': 'deposit', 'amount': 200, 'datetime': datetime.datetime(2026, 10, 4, 17, 46, 23, 510234), 'balance_after': 700.0, 'status': 'success'}
{'type': 'withdraw', 'amount': 100, 'datetime': datetime.datetime(2026, 10, 4, 17, 46, 23, 510264), 'balance_after': 600.0, 'status': 'success'}


Задание 2.
Этап 3. Реализация наследования
Реализуйте два класса CheckingAccount (расчётный счёт) и SavingsAccount (сберегательный счёт), которые отражают абстракцию базового поведения банковских аккаунтов:
* наследуются от базового класса Account;
* хранят атрибут класса account_type.

Класс SavingsAccount (сберегательный счёт) дополнительно должен реализовывать метод расчёта процентов на остаток apply_interest(self, rate) (например, 7% на остаток).
Класс SavingsAccount (сберегательный счёт) позволяет снимать деньги только до определенного порога баланса: нельзя снять больше 50% от баланса. Переопределите метод снятия со счёта.
Реализуйте валидацию на отрицательные суммы и корректность имени владельца:
* имя владельца счёта должно быть в формате «Имя Фамилия» с заглавных букв, кириллицей или латиницей, иначе — должно вызываться исключение;
* попытка положить отрицательную сумму должна вызывать исключение.
Реализуйте метод для анализа истории транзакций по размеру и дате:
* метод должен выводить последние n крупных операций.

In [9]:
import re

name_pattern = re.compile(r"^[A-ZА-ЯЁ][a-zа-яё]+ [A-ZА-ЯЁ][a-zа-яё]+$")


def validate_holder_name(name):
    if not name_pattern.fullmatch(name):
        raise ValueError("Имя владельца должно быть в формате 'Имя Фамилия'")


class CheckingAccount(Account):

    account_type = "checking"

    def __init__(self, account_holder: str, balance: float = 0):
        validate_holder_name(account_holder)
        super().__init__(account_holder, balance)


class SavingsAccount(Account):

    account_type = "savings"

    def __init__(self, account_holder: str, balance: float = 0):
        validate_holder_name(account_holder)
        super().__init__(account_holder, balance)

    def apply_interest(self, rate: float):
        if rate <= 0:
            raise ValueError("Ставка должна быть положительной.")

        interest = self._balance * rate
        self._balance += interest
        self._add_operation("interest", interest, "success")

    def withdraw(self, amount: float):
        if amount <= 0:
            self._add_operation("withdraw", amount, "fail")
            raise ValueError("Сумма снятия должна быть положительной.")

        if amount > self._balance * 0.5:
            self._add_operation("withdraw", amount, "fail")
            raise ValueError("Нельзя снять больше 50% от баланса.")

        self._balance -= amount
        self._add_operation("withdraw", amount, "success")


In [ ]:
# Тесты для проверки
checking = CheckingAccount("Иван Петров", 1000)
checking.deposit(300)
checking.withdraw(200)
print(checking.account_type, checking.get_balance())

savings = SavingsAccount("Anna Smith", 1000)
savings.apply_interest(0.07)
print(savings.get_balance())

try:
    savings.withdraw(600)
except ValueError as e:
    print(e)

savings.deposit(5000)
for op in savings.get_largest_operations(3):
    print(op)

try:
    CheckingAccount("иван петров", 100)
except ValueError as e:
    print(e)


checking 1100.0
1070.0
Нельзя снять больше 50% от баланса.
{'type': 'deposit', 'amount': 5000, 'datetime': datetime.datetime(2026, 10, 4, 17, 24, 20, 15043), 'balance_after': 6070.0, 'status': 'success'}
{'type': 'withdraw', 'amount': 600, 'datetime': datetime.datetime(2026, 10, 4, 17, 24, 20, 15017), 'balance_after': 1070.0, 'status': 'fail'}
{'type': 'interest', 'amount': 70.0, 'datetime': datetime.datetime(2026, 10, 4, 17, 24, 20, 14960), 'balance_after': 1070.0, 'status': 'success'}
Имя владельца должно быть в формате 'Имя Фамилия'


Задание 3. Задание для претендущих на оценку 8 и более баллов (выполняется по желанию).
Перед началом работы загрузите из личного кабинете файлы, на которых можно проверить код с «грязными» данными: transactions_dirty.csv и transactions_dirty.json.
Реализуйте для классов аккаунтов CheckingAccount (расчётный счёт) и SavingsAccount (сберегательный счёт) два метода:
* Метод загрузки истории в аккаунт из файла с транзакциями (файл транзакций общий для всех аккаунтов, необходимо учесть фильтрацию загружаемых значений).
* Метод clean_history(), который ищет ошибки в данных перед записью транзакций в историю (опечатки, отрицательные суммы, неверные даты). Обратите внимание, что для SavingsAccount (сберегательный счёт) доступно три типа операции (deposit, withdraw и interest), в то время как для CheckingAccount (расчётный счёт) доступны только два типа операции (deposit, withdraw). Все данные с ошибками считаем невалидными и не записываем в историю операций.
* После загрузки истории операций в аккаунт, баланс счёта должен обновиться.

In [ ]:
import csv
import json
from datetime import datetime

date_formats = ("%Y-%m-%d %H:%M:%S", "%d/%m/%Y %H:%M")

# Потому что в данных даты написаны в разных форматах
def parse_date(value):
    for fmt in date_formats:
        try:
            return datetime.strptime(value, fmt)
        except (ValueError, TypeError):
            continue
    return None


def read_transactions_file(filepath):
    if filepath.endswith(".json"):
        with open(filepath, encoding="utf-8") as f:
            return json.load(f)

    with open(filepath, encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f))


class CheckingAccount(Account):

    account_type = "checking"
    allowed_operations = {"deposit", "withdraw"}

    def __init__(self, account_holder: str, balance: float = 0):
        validate_holder_name(account_holder)
        super().__init__(account_holder, balance)

    def clean_history(self, records):
        valid_records = []
        for record in records:
            if record.get("operation") not in self.allowed_operations:
                continue

            try:
                amount = float(record.get("amount"))
            except (TypeError, ValueError):
                continue

            if amount <= 0:
                continue

            if parse_date(record.get("date")) is None:
                continue

            valid_records.append(record)

        return valid_records

    def load_transactions(self, filepath):
        records = read_transactions_file(filepath)
        own_records = [r for r in records if r.get("account_number") == self.account_number]
        valid_records = self.clean_history(own_records)
        valid_records.sort(key=lambda r: parse_date(r["date"]))

        for record in valid_records:
            amount = float(record["amount"])
            operation = record["operation"]

            if operation == "withdraw":
                self._balance -= amount
            else:
                self._balance += amount

            self.operations_history.append({
                "type": operation,
                "amount": amount,
                "datetime": parse_date(record["date"]),
                "balance_after": self._balance,
                "status": "success"
            })


class SavingsAccount(Account):

    account_type = "savings"
    allowed_operations = {"deposit", "withdraw", "interest"}

    def __init__(self, account_holder: str, balance: float = 0):
        validate_holder_name(account_holder)
        super().__init__(account_holder, balance)

    def apply_interest(self, rate: float):
        if rate <= 0:
            raise ValueError("Ставка должна быть положительной.")

        interest = self._balance * rate
        self._balance += interest
        self._add_operation("interest", interest, "success")

    def withdraw(self, amount: float):
        if amount <= 0:
            self._add_operation("withdraw", amount, "fail")
            raise ValueError("Сумма снятия должна быть положительной.")

        if amount > self._balance * 0.5:
            self._add_operation("withdraw", amount, "fail")
            raise ValueError("Нельзя снять больше 50% от баланса.")

        self._balance -= amount
        self._add_operation("withdraw", amount, "success")

    def clean_history(self, records):
        valid_records = []
        for record in records:
            if record.get("operation") not in self.allowed_operations:
                continue

            try:
                amount = float(record.get("amount"))
            except (TypeError, ValueError):
                continue

            if amount <= 0:
                continue

            if parse_date(record.get("date")) is None:
                continue

            valid_records.append(record)

        return valid_records

    def load_transactions(self, filepath):
        records = read_transactions_file(filepath)
        own_records = [r for r in records if r.get("account_number") == self.account_number]
        valid_records = self.clean_history(own_records)
        valid_records.sort(key=lambda r: parse_date(r["date"]))

        for record in valid_records:
            amount = float(record["amount"])
            operation = record["operation"]

            if operation == "withdraw":
                self._balance -= amount
            else:
                self._balance += amount

            self.operations_history.append({
                "type": operation,
                "amount": amount,
                "datetime": parse_date(record["date"]),
                "balance_after": self._balance,
                "status": "success"
            })


In [ ]:
# ТЕСТЫ ДЛЯ ПРОВЕРКИ
Account._account_counter = 100001

acc1 = CheckingAccount("Иван Петров", 1200)
acc2 = SavingsAccount("Anna Smith", 1500)
acc3 = CheckingAccount("Пётр Иванов", 800)
acc4 = SavingsAccount("Olga Orlova", 2000)

for acc in (acc1, acc2, acc3, acc4):
    acc.load_transactions("transactions_dirty.json")
    print(acc.account_number, acc.account_type, acc.get_balance(), len(acc.get_history()))

acc1_csv = CheckingAccount("Иван Петров", 1200)
Account._account_counter -= 1
acc1_csv.account_number = acc1.account_number
acc1_csv.load_transactions("transactions_dirty.csv")
print(acc1_csv.get_balance() == acc1.get_balance(), len(acc1_csv.get_history()) == len(acc1.get_history()))
